# Session start time zone

The published files in DANDI:000971 attach `pytz.timezone("US/Central")` with `datetime.replace`, which gives the zone's local mean time offset (-05:51) instead of CDT (-05:00) or CST (-06:00). This notebook reconverts one summer session and one winter session with the fixed code and compares their start times with the published files.

In [1]:
from datetime import datetime
from pathlib import Path
import shutil

import h5py
import pandas as pd

from lerner_lab_to_nwb.seiler_2024.seiler_2024_convert_dataset import (
    fp_to_nwb,
    get_no_port_entry_duration_sessions,
    get_session_key_from_kwargs,
)
from lerner_lab_to_nwb.seiler_2024.seiler_2024_convert_session import session_to_nwb

data_dir_path = Path("/Volumes/T7/CatalystNeuro/Lerner/_Raw_Data_for_future_reference")
published_dir_path = Path("/Volumes/T7/CatalystNeuro/Lerner/000971")
output_dir_path = Path("/Volumes/T7/CatalystNeuro/Lerner/fix_audit/fix_timezone")
if output_dir_path.exists():
    shutil.rmtree(output_dir_path)
output_dir_path.mkdir(parents=True)

In [2]:
session_to_nwb_kwargs_per_session = fp_to_nwb(
    data_dir_path=data_dir_path,
    output_dir_path=output_dir_path,
    start_variable="Start Date",
    stub_test=False,
    verbose=False,
)


def find_session(subject_id: str, start_datetime: datetime) -> dict:
    matches = [
        session_to_nwb_kwargs
        for session_to_nwb_kwargs in session_to_nwb_kwargs_per_session
        if session_to_nwb_kwargs["subject_id"] == subject_id
        and session_to_nwb_kwargs["session_conditions"].get("Start Date") == start_datetime.strftime("%m/%d/%y")
        and session_to_nwb_kwargs["session_conditions"].get("Start Time") == start_datetime.strftime("%H:%M:%S")
    ]
    assert len(matches) == 1, matches
    return matches[0]


sessions = [
    ("96.259", datetime(2019, 4, 9, 10, 34, 30)),  # daylight saving time
    ("64.205", datetime(2018, 11, 7, 10, 0, 25)),  # standard time
]
selected_kwargs = [find_session(subject_id, start_datetime) for subject_id, start_datetime in sessions]
no_port_entry_duration_sessions = get_no_port_entry_duration_sessions(
    port_entry_file_path=output_dir_path / "sessions_without_port_entry_durations.yaml",
    session_to_nwb_args_per_session=selected_kwargs,
)
for session_to_nwb_kwargs in selected_kwargs:
    if get_session_key_from_kwargs(session_to_nwb_kwargs) in no_port_entry_duration_sessions:
        session_to_nwb_kwargs["has_port_entry_durations"] = False
    session_to_nwb(**session_to_nwb_kwargs)

Reading port entry durations:   0%|          | 0/2 [00:00<?, ?it/s]

Reading port entry durations: 100%|██████████| 2/2 [00:00<00:00, 193.46it/s]

In [3]:
rows = []
for subject_id, start_datetime in sessions:
    session_stamp = start_datetime.isoformat().replace(":", "-")
    (published_file_path,) = (published_dir_path / f"sub-{subject_id.replace('.', '-')}").glob(f"*{session_stamp}*.nwb")
    (new_file_path,) = output_dir_path.glob(f"sub-{subject_id}_*{session_stamp}*.nwb")
    with h5py.File(published_file_path, "r") as published_file, h5py.File(new_file_path, "r") as new_file:
        published_start = pd.Timestamp(published_file["session_start_time"][()].decode())
        new_start = pd.Timestamp(new_file["session_start_time"][()].decode())
    rows.append(
        dict(
            subject_id=subject_id,
            published_start=published_start.isoformat(),
            new_start=new_start.isoformat(),
            shift_minutes=(new_start - published_start).total_seconds() / 60,
        )
    )
pd.DataFrame(rows)

,subject_id,published_start,new_start,shift_minutes
0,96.259,2019-04-09T10:34:30-05:51,2019-04-09T10:34:30-05:00,-51.0
1,64.205,2018-11-07T10:00:25-05:51,2018-11-07T10:00:25-06:00,9.0


The local wall-clock time is unchanged. The offset is now -05:00 in April and -06:00 in November, so the absolute start time moves 51 minutes earlier for the April session and 9 minutes later for the November one.